# 16-00 Speed-first Driving Behavior Gate

이 노트북의 목적은 11/11b/15에서 만든 양자화 후보를 **논문식 strict parity**가 아니라, 실제 주행에 가까운 **driving behavior gate**로 다시 평가하는 것이다.

중요한 점:

- 이 노트북은 “INT8 후보를 바로 주행에 투입해도 된다”를 선언하지 않는다.
- 이 노트북은 “Pi에서 다시 측정해볼 만한 후보”를 고르는 필터다.
- 기준 모델은 12번 FP32 ONNX이며, decoder와 driving postprocess는 10번 Pi runtime package의 standalone 구현을 그대로 쓴다.

지금 프로젝트에서 중요한 질문은 더 이상 “raw tensor가 원본 FP32와 거의 같은가?”만이 아니다.

중요한 질문은:

- 조향 방향이 반대로 뒤집히지 않는가?
- 곡선 구간에서 tangent heading이 같은 방향을 유지하는가?
- `single_left/right`, `lost/recovery` 판단이 위험하게 바뀌지 않는가?
- CPU latency를 줄일 만큼 빨라지는가?

기본 후보 범위는 `speed_shortlist_v2`다.  
이는 11/11b/15 결과 중 “속도 개선 가능성이 있거나, 이전 strict gate에서 near-miss였던 후보”만 먼저 본다.


## Gate 철학과 threshold 근거

기존 09/10/11/15 노트북의 gate는 주로 “원본 모델과 얼마나 같은가”를 봤다.

이번 노트북은 한 단계 덜 보수적으로 본다.

| 항목 | 이번 해석 |
|---|---|
| raw tensor diff | 참고용. 직접 pass/fail 기준으로 쓰지 않음 |
| lane pixel distance | 참고용. 큰 outlier frame을 찾는 데 사용 |
| heading sign flip | 중요. 확실히 좌/우로 향하는데 반대로 바뀌면 위험 |
| steer sign flip | 가장 중요. 확실한 조향 명령이 반대로 가면 실패 |
| single side flip | 매우 중요. `single_left` ↔ `single_right`는 safe offset 방향을 바꿀 수 있음 |
| mode mismatch | 중요. 단, 모든 mode mismatch가 동일하게 위험한 것은 아님 |
| latency | 매우 중요. Pi에서는 FP32가 이미 너무 느렸기 때문 |

작은 값 근처의 sign flip은 무시한다. 예를 들어 `steer=+0.01`과 `steer=-0.01`은 실제로 거의 직진이므로 위험하지 않다.

초기 threshold 근거:

- `STRONG_STEER_TH = 0.10`: 조향 명령의 작은 deadband. 이보다 작으면 거의 직진으로 취급한다.
- `STRONG_HEADING_TH = 0.08`: tangent slope `dx/dy` 기준이다. `atan(0.08)`은 약 4.6도라서, 이보다 작으면 heading sign flip을 강하게 벌하지 않는다.
- `p95` 기준: max는 일부 outlier 때문에 과하게 보수적일 수 있어, 자동 gate는 p95를 보되 top-risk overlay를 반드시 수동 확인한다.
- `single_side_flip_count`: 0이어야 한다. 한쪽 lane offset 방향이 뒤집히는 것은 주행에서 위험하다.

즉 이 노트북은 **자동 gate + 위험 프레임 수동 검수**를 한 세트로 본다.


In [1]:
from pathlib import Path
import os
import sys
import json
import time
import math
import platform
import shutil
from dataclasses import dataclass

import cv2
import numpy as np
import pandas as pd
import onnxruntime as ort

try:
    from scipy.interpolate import InterpolatedUnivariateSpline
    HAS_SCIPY = True
except Exception:
    InterpolatedUnivariateSpline = None
    HAS_SCIPY = False

print("python:", sys.executable)
print("machine:", platform.machine())
print("onnxruntime:", ort.__version__)
print("providers:", ort.get_available_providers())
print("HAS_SCIPY:", HAS_SCIPY)


python: ~\anaconda3\envs\<env>\python.exe
machine: AMD64
onnxruntime: 1.23.2
providers: ['AzureExecutionProvider', 'CPUExecutionProvider']
HAS_SCIPY: True


In [2]:
# ----- Experiment paths -----
PROJECT_ROOT12 = Path(r"~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild")
PROJECT_ROOT15 = Path(r"~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\15_clrkdnet_qat_int8_recovery")
PROJECT_ROOT16 = Path(r"~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\16_quantized_driving_behavior_gate")

PACKAGE_ROOT = PROJECT_ROOT12 / "review_outputs" / "10_pi_runtime_latency_sequence_validation_v1" / "pkg"
OUT_ROOT = PROJECT_ROOT16 / "review_outputs" / "00_speed_first_driving_behavior_gate_v1"
TABLE_DIR = OUT_ROOT / "tables"
VIS_DIR = OUT_ROOT / "visuals"
for p in [PROJECT_ROOT16 / "notebooks", OUT_ROOT, TABLE_DIR, VIS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Geometry / contracts from 05-10.
RAW_W = 1296
RAW_H = 972
CUT_HEIGHT = 445
IMG_W = 800
IMG_H = 320
NUM_POINTS = 72
N_STRIPS = NUM_POINTS - 1
N_OFFSETS = NUM_POINTS
NUM_PRIORS = 192
OUTPUT_DIM = 78
SAMPLE_Y = list(range(971, 444, -20))
IMAGE_CENTER_X = RAW_W / 2.0

DEFAULT_PAIR_BONUS_PX = 60.0
ORT_WARMUP_RUNS = 5

# Run controls.
MAX_SEQUENCE_FRAMES = None  # None이면 10번 pkg의 field3 sequence 전체(보통 240장)를 사용한다.
LOCAL_THREADS = None        # None이면 onnxruntime 기본값. Pi용 최종 판단은 Pi에서 threads=2/3으로 다시 측정한다.
CANDIDATE_SCOPE = "speed_shortlist_v2"  # "speed_shortlist_v2" 또는 "all_reviewed"

# Driving behavior gate thresholds. 필요하면 여기만 조정한다.
STRONG_STEER_TH = 0.10
STRONG_HEADING_TH = 0.08
STEER_ABS_DIFF_P95_LIMIT = 0.18
HEADING_ABS_DIFF_P95_LIMIT = 0.20
MODE_MISMATCH_RATIO_LIMIT = 0.15
EXTRA_LARGE_JUMP_LIMIT = 3
LARGE_STEER_JUMP_TH = 0.25

BASELINE = {
    "name": "original_fp32",
    "family": "baseline",
    "path": PROJECT_ROOT12 / "review_outputs" / "09_onnx_export_parity_v1" / "models" / "MapLane_LocalFit_Field12_v1_best_opset17_static_b1.onnx",
}

ALL_CANDIDATES = [
    # 11: full graph PTQ candidates.
    {"name": "ptq11_static_qdq_s8s8", "family": "11_full_ptq", "path": PROJECT_ROOT12 / "review_outputs" / "11_onnx_cpu_quantization_candidates_v1" / "models" / "static_qdq_s8s8.onnx"},
    {"name": "ptq11_static_qdq_u8s8", "family": "11_full_ptq", "path": PROJECT_ROOT12 / "review_outputs" / "11_onnx_cpu_quantization_candidates_v1" / "models" / "static_qdq_u8s8.onnx"},
    {"name": "ptq11_static_qoperator_u8s8", "family": "11_full_ptq", "path": PROJECT_ROOT12 / "review_outputs" / "11_onnx_cpu_quantization_candidates_v1" / "models" / "static_qoperator_u8s8.onnx"},
    {"name": "ptq11_static_qoperator_u8u8", "family": "11_full_ptq", "path": PROJECT_ROOT12 / "review_outputs" / "11_onnx_cpu_quantization_candidates_v1" / "models" / "static_qoperator_u8u8.onnx"},

    # 11b: selective PTQ candidates.
    {"name": "ptq11b_backbone_layer4_qdq_u8s8", "family": "11b_selective_ptq", "path": PROJECT_ROOT12 / "review_outputs" / "11b_onnx_selective_quantization_boundary_v1" / "models" / "backbone_layer4_qdq_u8s8.onnx"},
    {"name": "ptq11b_backbone_layer3_layer4_qdq_u8s8", "family": "11b_selective_ptq", "path": PROJECT_ROOT12 / "review_outputs" / "11b_onnx_selective_quantization_boundary_v1" / "models" / "backbone_layer3_layer4_qdq_u8s8.onnx"},
    {"name": "ptq11b_backbone_all_qdq_u8s8", "family": "11b_selective_ptq", "path": PROJECT_ROOT12 / "review_outputs" / "11b_onnx_selective_quantization_boundary_v1" / "models" / "backbone_all_qdq_u8s8.onnx"},
    {"name": "ptq11b_backbone_neck_qdq_u8s8", "family": "11b_selective_ptq", "path": PROJECT_ROOT12 / "review_outputs" / "11b_onnx_selective_quantization_boundary_v1" / "models" / "backbone_neck_qdq_u8s8.onnx"},
    {"name": "ptq11b_backbone_all_qoperator_u8s8", "family": "11b_selective_ptq", "path": PROJECT_ROOT12 / "review_outputs" / "11b_onnx_selective_quantization_boundary_v1" / "models" / "backbone_all_qoperator_u8s8.onnx"},

    # 15: QAT-lite + full static QDQ candidates.
    {"name": "qat15_layer4_only_static_qdq_u8s8", "family": "15_qat_full_static", "path": PROJECT_ROOT15 / "models" / "int8_onnx" / "qat_layer4_only_static_qdq_u8s8.onnx"},
    {"name": "qat15_full_model_static_qdq_u8s8", "family": "15_qat_full_static", "path": PROJECT_ROOT15 / "models" / "int8_onnx" / "qat_full_model_static_qdq_u8s8.onnx"},
    {"name": "qat15_backbone_only_static_qdq_u8s8", "family": "15_qat_full_static", "path": PROJECT_ROOT15 / "models" / "int8_onnx" / "qat_backbone_only_static_qdq_u8s8.onnx"},
    {"name": "qat15_backbone_neck_only_static_qdq_u8s8", "family": "15_qat_full_static", "path": PROJECT_ROOT15 / "models" / "int8_onnx" / "qat_backbone_neck_only_static_qdq_u8s8.onnx"},
]

SPEED_SHORTLIST_NAMES = {
    # fastest from 11/15, plus 11b candidates that may reduce size/CPU enough to matter.
    "ptq11_static_qdq_u8s8",
    "ptq11_static_qoperator_u8s8",
    "ptq11b_backbone_layer4_qdq_u8s8",  # quality anchor: best previous meaning preservation, though not fast.
    "ptq11b_backbone_all_qdq_u8s8",
    "ptq11b_backbone_neck_qdq_u8s8",
    "ptq11b_backbone_all_qoperator_u8s8",
    "qat15_layer4_only_static_qdq_u8s8",
    "qat15_full_model_static_qdq_u8s8",
}

if CANDIDATE_SCOPE == "all_reviewed":
    CANDIDATES = ALL_CANDIDATES
elif CANDIDATE_SCOPE == "speed_shortlist_v2":
    CANDIDATES = [c for c in ALL_CANDIDATES if c["name"] in SPEED_SHORTLIST_NAMES]
else:
    raise ValueError(f"Unknown CANDIDATE_SCOPE: {CANDIDATE_SCOPE}")

for item in [BASELINE] + CANDIDATES:
    assert item["path"].exists(), item["path"]
    print(item["name"], item["path"], f"{item['path'].stat().st_size / (1024*1024):.2f} MB")

print("CANDIDATE_SCOPE:", CANDIDATE_SCOPE)
print("candidate count:", len(CANDIDATES))
print("OUT_ROOT:", OUT_ROOT)


original_fp32 ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\09_onnx_export_parity_v1\models\MapLane_LocalFit_Field12_v1_best_opset17_static_b1.onnx 0.35 MB
ptq11_static_qdq_u8s8 ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\11_onnx_cpu_quantization_candidates_v1\models\static_qdq_u8s8.onnx 11.52 MB
ptq11_static_qoperator_u8s8 ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\11_onnx_cpu_quantization_candidates_v1\models\static_qoperator_u8s8.onnx 11.40 MB
ptq11b_backbone_layer4_qdq_u8s8 ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\11b_onnx_selective_quantization_boundary_v1\models\backbone_layer4_qdq_u8s8.onnx 20.29 MB
ptq11b_backbone_all_qdq_u8s8 ~\02_Projects\Unive

## 10번 runtime core 재사용

아래 utility와 runtime core는 10번 Pi 검증 노트북에서 사용한 standalone decoder/steering 구현을 그대로 가져온 것이다. 따라서 이번 평가도 Pi 배포 스크립트의 decode/steering과 같은 의미를 가진다.


In [3]:
# ----- Robust filesystem helpers -----
def fs_path(path):
    p = Path(path)
    s = str(p.resolve())
    if sys.platform.startswith("win") and not s.startswith("\\\\?\\"):
        return "\\\\?\\" + s
    return s

def exists_fs(path):
    return os.path.exists(fs_path(path))

def read_json(path):
    with open(fs_path(path), "r", encoding="utf-8") as f:
        return json.load(f)

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(fs_path(path), "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def write_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(fs_path(path), "w", encoding="utf-8-sig", newline="") as f:
        df.to_csv(f, index=False)

def copy_file(src, dst):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    return shutil.copy2(fs_path(src), fs_path(dst))

def imread_bgr(path):
    data = np.fromfile(fs_path(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    return img

def imwrite_bgr(path, image):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ok, buf = cv2.imencode(path.suffix or ".jpg", image)
    if not ok:
        raise RuntimeError(f"Could not encode image: {path}")
    buf.tofile(fs_path(path))
    return path

def safe_key(prefix, source_path, order):
    h = hashlib.sha1(str(source_path).replace("\\", "/").encode("utf-8")).hexdigest()[:10]
    return f"{prefix}_{int(order):04d}_{h}"

def even_sample(items, n):
    items = list(items)
    if n is None or len(items) <= n:
        return items
    idx = np.linspace(0, len(items) - 1, n).round().astype(int)
    out, seen = [], set()
    for i in idx:
        i = int(i)
        if i not in seen:
            out.append(items[i])
            seen.add(i)
    return out

def percentile_summary(values):
    arr = np.asarray(list(values), dtype=np.float64)
    if len(arr) == 0:
        return {"mean": None, "p50": None, "p90": None, "p95": None, "max": None}
    return {
        "mean": float(arr.mean()),
        "p50": float(np.percentile(arr, 50)),
        "p90": float(np.percentile(arr, 90)),
        "p95": float(np.percentile(arr, 95)),
        "max": float(arr.max()),
    }

In [4]:
# ----- Contracts and ONNX Runtime -----
def load_package_contracts(package_root):
    package_root = Path(package_root)
    decode_contract = read_json(package_root / "c" / "decode_contract_v1.json")
    driving_contract = read_json(package_root / "c" / "driving_contract_v1.json")
    driving_contract["driving_contract"]["params"].setdefault("pair_bonus_px", DEFAULT_PAIR_BONUS_PX)
    onnx_report = read_json(package_root / "c" / "onnx_parity_report_v1.json")
    model_info = read_json(package_root / "m" / "model_path.json")
    return decode_contract, driving_contract, onnx_report, model_info

def make_ort_session(package_root, intra_op_num_threads=None):
    package_root = Path(package_root)
    _, _, _, model_info = load_package_contracts(package_root)
    model_path = package_root / model_info["onnx_rel"]
    data_path = package_root / model_info["external_data_rel"]
    assert exists_fs(model_path), model_path
    assert exists_fs(data_path), data_path

    if intra_op_num_threads is None and IS_PI:
        intra_op_num_threads = DEFAULT_PI_ORT_THREADS
    so = ort.SessionOptions()
    if intra_op_num_threads is not None:
        so.intra_op_num_threads = int(intra_op_num_threads)
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    session = ort.InferenceSession(fs_path(model_path), sess_options=so, providers=["CPUExecutionProvider"])
    return session, session.get_inputs()[0].name, session.get_outputs()[0].name

def preprocess_bgr_for_model(bgr):
    crop = bgr[int(CUT_HEIGHT):, :, :]
    resized = cv2.resize(crop, (IMG_W, IMG_H), interpolation=cv2.INTER_LINEAR)
    tensor = resized.astype(np.float32).transpose(2, 0, 1)[None, ...] / 255.0
    assert tensor.shape == (1, 3, IMG_H, IMG_W)
    return tensor

def run_onnx_raw(session, input_name, output_name, bgr):
    inp = preprocess_bgr_for_model(bgr)
    out = session.run([output_name], {input_name: inp})[0]
    assert out.shape == (1, NUM_PRIORS, OUTPUT_DIM), out.shape
    return out[0].astype(np.float32)

def warmup_ort_session(session, input_name, output_name, package_root, records, label="ORT"):
    if ORT_WARMUP_RUNS <= 0 or not records:
        return
    first = records[0]
    bgr = imread_bgr(Path(package_root) / first["image_rel"])
    inp = preprocess_bgr_for_model(bgr)
    for _ in range(int(ORT_WARMUP_RUNS)):
        _ = session.run([output_name], {input_name: inp})[0]
    print(f"{label} warmup runs:", ORT_WARMUP_RUNS)

# ----- 07 standalone decoder -----
def softmax_positive_score(logits_2):
    logits = logits_2.astype(np.float32)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp[:, 1] / exp.sum(axis=1)

def official_cuda_suppresses(a, b, threshold):
    start_a = int(float(a[2]) * N_STRIPS + 0.5)
    start_b = int(float(b[2]) * N_STRIPS + 0.5)
    start = max(start_a, start_b)
    len_a, len_b = float(a[4]), float(b[4])
    end_a = int(start_a + len_a - 1 + 0.5 - (1 if (len_a - 1) < 0 else 0))
    end_b = int(start_b + len_b - 1 + 0.5 - (1 if (len_b - 1) < 0 else 0))
    end = min(end_a, end_b, N_OFFSETS - 1)
    if end < start:
        return False
    dist = float(np.abs(a[5 + start:5 + end + 1] - b[5 + start:5 + end + 1]).sum())
    return dist < float(threshold) * float(end - start + 1)

def official_overlap_nms(nms_predictions, scores, nms_thres, top_k):
    order = np.argsort(-scores)
    kept = []
    for idx in order:
        duplicate = any(official_cuda_suppresses(nms_predictions[idx], nms_predictions[j], nms_thres) for j in kept)
        if not duplicate:
            kept.append(int(idx))
        if len(kept) >= int(top_k):
            break
    return kept

def resample_lane_xs(xs, ys, query_ys):
    if len(xs) < 2:
        return np.full_like(query_ys, -2.0, dtype=np.float32)
    order = np.argsort(ys)
    ys_sorted, xs_sorted = ys[order], xs[order]
    spline = InterpolatedUnivariateSpline(ys_sorted, xs_sorted, k=min(3, len(xs_sorted) - 1))
    out = spline(query_ys).astype(np.float32)
    min_y, max_y = float(ys_sorted.min()) - 0.01, float(ys_sorted.max()) + 0.01
    out[(query_ys < min_y) | (query_ys > max_y)] = -2.0
    return out

def prediction_to_lane_array(prediction, confidence):
    pred = prediction.copy().astype(np.float32)
    lane_xs = pred[6:].copy()
    start = min(max(0, int(round(float(pred[2]) * N_STRIPS))), N_STRIPS)
    length = int(round(float(pred[5])))
    end = min(start + length - 1, N_OFFSETS - 1)
    lane_xs[end + 1:] = -2.0
    if start > 0:
        valid_prefix = ((lane_xs[:start] >= 0.0) & (lane_xs[:start] <= 1.0)).astype(np.int32)
        mask = ~(valid_prefix[::-1].cumprod()[::-1].astype(bool))
        prefix = lane_xs[:start].copy()
        prefix[mask] = -2.0
        lane_xs[:start] = prefix

    prior_ys = np.linspace(1.0, 0.0, N_OFFSETS, dtype=np.float32)
    valid = lane_xs >= 0.0
    if int(valid.sum()) <= 1:
        return None
    xs_norm = lane_xs[valid][::-1].astype(np.float32)
    ys_crop_norm = prior_ys[valid][::-1].astype(np.float32)
    ys_norm = (ys_crop_norm * float(RAW_H - CUT_HEIGHT) + float(CUT_HEIGHT)) / float(RAW_H)

    sample_ys_norm = np.array(SAMPLE_Y, dtype=np.float32) / float(RAW_H)
    sample_xs_norm = resample_lane_xs(xs_norm, ys_norm, sample_ys_norm)
    valid_sample = (sample_xs_norm >= 0.0) & (sample_xs_norm < 1.0)
    if int(valid_sample.sum()) <= 1:
        return None
    pts = np.stack([sample_xs_norm[valid_sample] * float(RAW_W), sample_ys_norm[valid_sample] * float(RAW_H)], axis=1)
    return {"points": pts.astype(np.float32), "conf": float(confidence)}

def decode_raw_to_lanes(raw_predictions, decode_contract):
    dc = decode_contract.get("decoder_contract", decode_contract)
    conf_threshold = float(dc["conf_threshold"])
    nms_thres = float(dc["nms_thres"])
    nms_topk = int(dc["nms_topk"])
    predictions = raw_predictions.astype(np.float32)
    scores = softmax_positive_score(predictions[:, :2])
    keep_mask = scores >= conf_threshold
    if int(keep_mask.sum()) == 0:
        return []
    pred_kept, score_kept = predictions[keep_mask].copy(), scores[keep_mask].copy()
    nms_predictions = np.concatenate([pred_kept[:, :4], pred_kept[:, 5:]], axis=1).astype(np.float32)
    nms_predictions[:, 4] *= float(N_STRIPS)
    nms_predictions[:, 5:] *= float(IMG_W - 1)
    keep = official_overlap_nms(nms_predictions, score_kept, nms_thres, nms_topk)
    selected, selected_scores = pred_kept[keep].copy(), score_kept[keep].copy()
    selected[:, 5] = np.round(selected[:, 5] * float(N_STRIPS))
    lanes = []
    for pred, score in zip(selected, selected_scores):
        lane = prediction_to_lane_array(pred, score)
        if lane is not None:
            lanes.append(lane)
    return lanes

def lanes_to_jsonable(lanes):
    return [{"conf": float(l.get("conf", 0.0)), "points": np.asarray(l["points"], dtype=float).round(4).tolist()} for l in lanes]

def lanes_from_jsonable(items):
    return [{"conf": float(x.get("conf", 0.0)), "points": np.asarray(x["points"], dtype=np.float32)} for x in items]

# ----- 08 inside_soft steering -----
def interp_or_nearest_x(points, y, max_y_distance):
    pts = np.asarray(points, dtype=np.float32)
    if len(pts) == 0:
        return None
    xs, ys = pts[:, 0], pts[:, 1]
    order = np.argsort(ys)
    ys, xs = ys[order], xs[order]
    if float(ys.min()) <= y <= float(ys.max()) and len(pts) >= 2:
        return float(np.interp(y, ys, xs))
    nearest_idx = int(np.argmin(np.abs(ys - y)))
    if abs(float(ys[nearest_idx]) - float(y)) <= float(max_y_distance):
        return float(xs[nearest_idx])
    return None

def feature_from_lane(lane, pp):
    pts = np.asarray(lane["points"], dtype=np.float32)
    if len(pts) < int(pp["min_points"]):
        return None
    if float(pts[:, 1].max() - pts[:, 1].min()) < float(pp["min_y_span"]):
        return None
    y_top = float((RAW_H - 1) * pp["top_ratio"])
    y_primary = float((RAW_H - 1) * pp["primary_ratio"])
    y_bottom = float((RAW_H - 1) * pp["bottom_ratio"])
    x_top = interp_or_nearest_x(pts, y_top, pp["max_y_distance"])
    x_primary = interp_or_nearest_x(pts, y_primary, pp["max_y_distance"])
    x_bottom = interp_or_nearest_x(pts, y_bottom, pp["max_y_distance"])
    if x_top is None or x_primary is None or x_bottom is None:
        return None
    return {"x_top": x_top, "x_primary": x_primary, "x_bottom": x_bottom, "heading": (x_top - x_bottom) / max(1.0, y_bottom - y_top), "y_top": y_top, "y_primary": y_primary, "y_bottom": y_bottom}

def estimate_steer(center_x, heading, pp):
    pos_error = (float(center_x) - IMAGE_CENTER_X) / IMAGE_CENTER_X
    raw = float(pp["steer_gain"]) * (float(pp["k_pos"]) * pos_error + float(pp["k_heading"]) * float(heading))
    return float(np.clip(raw, -float(pp["max_steer_norm"]), float(pp["max_steer_norm"])))

def trajectory_from_pair(left, right):
    center_top = 0.5 * (left["x_top"] + right["x_top"])
    center_primary = 0.5 * (left["x_primary"] + right["x_primary"])
    center_bottom = 0.5 * (left["x_bottom"] + right["x_bottom"])
    return {"center_primary": center_primary, "heading": (center_top - center_bottom) / max(1.0, left["y_bottom"] - left["y_top"]), "mode": "both_stable"}

def trajectory_from_single(feat, side, pp):
    sign = 1.0 if side == "left" else -1.0
    offset = float(pp["safe_offset_px"])
    center_top = feat["x_top"] + sign * offset
    center_primary = feat["x_primary"] + sign * offset
    center_bottom = feat["x_bottom"] + sign * offset
    return {"center_primary": center_primary, "heading": (center_top - center_bottom) / max(1.0, feat["y_bottom"] - feat["y_top"]), "mode": f"single_{side}", "source_x": feat["x_primary"]}

def init_drive_memory():
    return {"smoothed_center_x": IMAGE_CENTER_X, "smoothed_heading": 0.0, "last_steer_norm": 0.0, "turn_bias": 0.0, "lost_frames": 0, "unstable_frames": 0}

def choose_trajectory(features, memory, pp):
    if not features:
        return None
    lefts = [f for f in features if f["x_primary"] < IMAGE_CENTER_X]
    rights = [f for f in features if f["x_primary"] >= IMAGE_CENTER_X]
    left = max(lefts, key=lambda f: f["x_primary"]) if lefts else None
    right = min(rights, key=lambda f: f["x_primary"]) if rights else None
    candidates = []
    if left is not None and right is not None:
        gap = right["x_primary"] - left["x_primary"]
        if float(pp["gap_min_px"]) <= gap <= float(pp["gap_max_px"]):
            candidates.append(trajectory_from_pair(left, right))
    for feat in features:
        for side in ["left", "right"]:
            cand = trajectory_from_single(feat, side, pp)
            penalty = 0.0
            if feat["x_primary"] < IMAGE_CENTER_X - float(pp["single_side_deadband_px"]) and side == "right":
                penalty = float(pp["side_prior_penalty_px"])
            if feat["x_primary"] > IMAGE_CENTER_X + float(pp["single_side_deadband_px"]) and side == "left":
                penalty = float(pp["side_prior_penalty_px"])
            cand["side_penalty"] = penalty
            candidates.append(cand)
    prev_center = memory.get("smoothed_center_x", IMAGE_CENTER_X)
    prev_heading = memory.get("smoothed_heading", 0.0)
    prev_steer = memory.get("last_steer_norm", 0.0)
    def cost(c):
        center_cost = abs(c["center_primary"] - prev_center)
        heading_cost = abs(c["heading"] - prev_heading) * float(pp["heading_cost_px"])
        steer_cost = abs(estimate_steer(c["center_primary"], c["heading"], pp) - prev_steer) * float(pp["steer_cost_px"])
        pair_bonus = -float(pp.get("pair_bonus_px", DEFAULT_PAIR_BONUS_PX)) if c["mode"] == "both_stable" else 0.0
        return center_cost + heading_cost + steer_cost + c.get("side_penalty", 0.0) + pair_bonus
    best = min(candidates, key=cost)
    best["candidate_count"] = len(candidates)
    best["candidate_cost"] = float(cost(best))
    return best

def update_drive(lanes, memory, driving_contract):
    pp = driving_contract["driving_contract"]["params"] if "driving_contract" in driving_contract else driving_contract["params"]
    features = [f for f in (feature_from_lane(l, pp) for l in lanes) if f is not None]
    measured = choose_trajectory(features, memory, pp)
    raw_mode = "lost" if measured is None else measured["mode"]
    if measured is None:
        memory["lost_frames"] += 1
        memory["unstable_frames"] = 0
        if memory["lost_frames"] <= int(pp["lost_short_frames"]):
            mode = "lost_short_recovery"
            target = memory["last_steer_norm"] * float(pp["recovery_decay"])
        else:
            mode = "lost_active_recovery"
            sign = float(np.sign(memory["turn_bias"])) or float(np.sign(memory["last_steer_norm"]))
            ramp = float(pp["recovery_ramp"]) * max(0, memory["lost_frames"] - int(pp["lost_short_frames"]))
            target = memory["last_steer_norm"] * float(pp["recovery_decay"]) + float(pp["recovery_gain"]) * memory["turn_bias"] + ramp * sign
        target = float(np.clip(target, -float(pp["max_steer_norm"]), float(pp["max_steer_norm"])))
        memory["smoothed_heading"] *= float(pp["lost_heading_decay"])
        steer = float(pp["steer_alpha"]) * target + (1.0 - float(pp["steer_alpha"])) * memory["last_steer_norm"]
        steer = float(np.clip(steer, -float(pp["max_steer_norm"]), float(pp["max_steer_norm"])))
        memory["last_steer_norm"] = steer
        memory["turn_bias"] = float(pp["turn_bias_alpha"]) * steer + (1.0 - float(pp["turn_bias_alpha"])) * memory["turn_bias"]
        return {"raw_mode": raw_mode, "effective_mode": mode, "feature_count": len(features), "candidate_count": 0, "measured_center_x": np.nan, "measured_heading": np.nan, "smoothed_center_x": memory["smoothed_center_x"], "smoothed_heading": memory["smoothed_heading"], "steer_norm": steer, "turn_bias": memory["turn_bias"], "lost_frames": memory["lost_frames"]}

    memory["lost_frames"] = 0
    center, heading = float(measured["center_primary"]), float(measured["heading"])
    center_jump = abs(center - memory["smoothed_center_x"])
    heading_jump = abs(heading - memory["smoothed_heading"])
    unstable = center_jump > float(pp["jump_center_px"]) or heading_jump > float(pp["jump_heading"])
    if unstable:
        memory["unstable_frames"] += 1
        blend = min(float(pp["jump_blend_max"]), float(pp["jump_blend_start"]) + float(pp["jump_blend_step"]) * max(0, memory["unstable_frames"] - 1))
        effective_center = (1.0 - blend) * memory["smoothed_center_x"] + blend * center
        effective_heading = (1.0 - blend) * memory["smoothed_heading"] + blend * heading
        mode = "unstable_blend"
    else:
        memory["unstable_frames"] = 0
        effective_center, effective_heading, mode = center, heading, measured["mode"]
    memory["smoothed_center_x"] = float(pp["center_alpha"]) * effective_center + (1.0 - float(pp["center_alpha"])) * memory["smoothed_center_x"]
    memory["smoothed_heading"] = float(pp["heading_alpha"]) * effective_heading + (1.0 - float(pp["heading_alpha"])) * memory["smoothed_heading"]
    target = estimate_steer(memory["smoothed_center_x"], memory["smoothed_heading"], pp)
    steer = float(pp["steer_alpha"]) * target + (1.0 - float(pp["steer_alpha"])) * memory["last_steer_norm"]
    steer = float(np.clip(steer, -float(pp["max_steer_norm"]), float(pp["max_steer_norm"])))
    memory["last_steer_norm"] = steer
    memory["turn_bias"] = float(pp["turn_bias_alpha"]) * steer + (1.0 - float(pp["turn_bias_alpha"])) * memory["turn_bias"]
    return {"raw_mode": raw_mode, "effective_mode": mode, "feature_count": len(features), "candidate_count": int(measured.get("candidate_count", 0)), "measured_center_x": center, "measured_heading": heading, "smoothed_center_x": memory["smoothed_center_x"], "smoothed_heading": memory["smoothed_heading"], "steer_norm": steer, "turn_bias": memory["turn_bias"], "lost_frames": memory["lost_frames"]}

## STOP CHECK A: 경로, 계약, 데이터 고정

이 셀은 실험 전제 자체를 확인한다.

- 10번 package의 decoder/driving contract를 사용하고 있는가?
- field3 sequence가 같은 이미지 순서로 고정되어 있는가?
- 후보 ONNX와 external data 파일이 실제로 존재하는가?
- 후보 이름이 중복되지 않는가?

이 확인이 깨지면 이후 숫자는 비교 의미가 없다.


In [5]:
def onnx_external_data_ok(path):
    path = Path(path)
    if path.stat().st_size < 1024 * 1024:
        return path.with_name(path.name + ".data").exists()
    return True

decode_contract_probe = read_json(PACKAGE_ROOT / "c" / "decode_contract_v1.json")
driving_contract_probe = read_json(PACKAGE_ROOT / "c" / "driving_contract_v1.json")
records_probe = pd.read_csv(PACKAGE_ROOT / "t" / "records_manifest.csv")
seq_probe = records_probe[records_probe["role"] == "sequence"].sort_values("order").reset_index(drop=True)

assert decode_contract_probe["decoder_contract"]["name"] == "official_overlap_python"
assert abs(float(decode_contract_probe["decoder_contract"]["conf_threshold"]) - 0.35) < 1e-9
assert abs(float(decode_contract_probe["decoder_contract"]["nms_thres"]) - 70.0) < 1e-9
assert "driving_contract" in driving_contract_probe
assert len(seq_probe) > 0
assert seq_probe["key"].is_unique
assert all((PACKAGE_ROOT / p).exists() for p in seq_probe["image_rel"].head(5))
assert len({c["name"] for c in CANDIDATES}) == len(CANDIDATES)
for item in [BASELINE] + CANDIDATES:
    assert item["path"].exists(), item["path"]
    assert onnx_external_data_ok(item["path"]), item["path"]

stop_check = {
    "decode": decode_contract_probe["decoder_contract"],
    "driving_name": driving_contract_probe["driving_contract"].get("name"),
    "sequence_frames": int(len(seq_probe)),
    "candidate_scope": CANDIDATE_SCOPE,
    "candidate_count": len(CANDIDATES),
}
print(json.dumps(stop_check, indent=2, ensure_ascii=False))
print("STOP CHECK A passed.")


{
  "decode": {
    "name": "official_overlap_python",
    "conf_threshold": 0.35,
    "nms_thres": 70.0,
    "nms_topk": 4,
    "source": "Python port of clrkd/ops/csrc/nms_kernel.cu devIoU + greedy keep order",
    "status": "selected_after_full_val_sweep",
    "selection_rule": "highest val proxy F1, tie-broken by precision then recall",
    "confidence_metadata": "lane.metadata[\"conf\"] is stored as softmax positive-lane score in [0, 1], not raw cls logit."
  },
  "driving_name": "inside_soft",
  "sequence_frames": 240,
  "candidate_scope": "speed_shortlist_v2",
  "candidate_count": 8
}
STOP CHECK A passed.


In [6]:
# ----- Arbitrary model session helper -----
def make_ort_session_for_model(model_path, intra_op_num_threads=None):
    model_path = Path(model_path)
    assert exists_fs(model_path), model_path
    data_path = model_path.with_name(model_path.name + ".data")
    # External data file is required for the original FP32 ONNX. INT8 candidates may not have one.
    if model_path.name.endswith(".onnx") and model_path.stat().st_size < 1024 * 1024:
        assert exists_fs(data_path), f"Small ONNX graph likely needs external data: {data_path}"

    so = ort.SessionOptions()
    if intra_op_num_threads is not None:
        so.intra_op_num_threads = int(intra_op_num_threads)
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    session = ort.InferenceSession(fs_path(model_path), sess_options=so, providers=["CPUExecutionProvider"])
    return session, session.get_inputs()[0].name, session.get_outputs()[0].name

def run_session_raw_from_tensor(session, input_name, output_name, tensor):
    out = session.run([output_name], {input_name: tensor})[0]
    assert out.shape == (1, NUM_PRIORS, OUTPUT_DIM), out.shape
    return out[0].astype(np.float32)

decode_contract, driving_contract, _, _ = load_package_contracts(PACKAGE_ROOT)
records = pd.read_csv(PACKAGE_ROOT / "t" / "records_manifest.csv")
seq_records = records[records["role"] == "sequence"].sort_values("order").reset_index(drop=True)
if MAX_SEQUENCE_FRAMES is not None:
    seq_records = seq_records.iloc[:int(MAX_SEQUENCE_FRAMES)].copy()

print("sequence frames:", len(seq_records))
display(seq_records.head())


sequence frames: 240


,key,set,role,order,image_rel,source_path_local,source_name
0,f3s_0000_c0bbf9d393,field3,sequence,0,i/f3s_0000_c0bbf9d393.jpg,~\02_Projects\University\26-1_Em...,000000_mixed_drive_auto_20260501_191234_792.jpg
1,f3s_0001_ad507632de,field3,sequence,1,i/f3s_0001_ad507632de.jpg,~\02_Projects\University\26-1_Em...,000007_mixed_drive_auto_20260501_191236_421.jpg
2,f3s_0002_5001e69f43,field3,sequence,2,i/f3s_0002_5001e69f43.jpg,~\02_Projects\University\26-1_Em...,000014_mixed_drive_auto_20260501_191237_989.jpg
3,f3s_0003_ab11851eee,field3,sequence,3,i/f3s_0003_ab11851eee.jpg,~\02_Projects\University\26-1_Em...,000022_mixed_drive_auto_20260501_191239_769.jpg
4,f3s_0004_0befa0c953,field3,sequence,4,i/f3s_0004_0befa0c953.jpg,~\02_Projects\University\26-1_Em...,000029_mixed_drive_auto_20260501_191241_242.jpg


## 후보별 sequence 실행

각 모델에 field3 sequence를 넣고 아래를 저장한다.

- decoded lane
- driving postprocess 결과
- latency 분해

baseline인 원본 FP32와 후보 INT8을 **같은 이미지, 같은 decoder, 같은 steering memory 초기값**으로 실행한다.


In [7]:
def run_model_sequence(model_info, records_df):
    session, input_name, output_name = make_ort_session_for_model(model_info["path"], LOCAL_THREADS)
    # warmup
    warm_bgr = imread_bgr(PACKAGE_ROOT / records_df.iloc[0]["image_rel"])
    warm_tensor = preprocess_bgr_for_model(warm_bgr)
    for _ in range(ORT_WARMUP_RUNS):
        _ = run_session_raw_from_tensor(session, input_name, output_name, warm_tensor)

    memory = init_drive_memory()
    rows = []
    decoded_by_key = {}
    for idx, rec in records_df.iterrows():
        key = rec["key"]
        image_path = PACKAGE_ROOT / rec["image_rel"]
        t0 = time.perf_counter()
        bgr = imread_bgr(image_path)
        t1 = time.perf_counter()
        tensor = preprocess_bgr_for_model(bgr)
        t2 = time.perf_counter()
        raw = run_session_raw_from_tensor(session, input_name, output_name, tensor)
        t3 = time.perf_counter()
        lanes = decode_raw_to_lanes(raw, decode_contract)
        t4 = time.perf_counter()
        drive = update_drive(lanes, memory, driving_contract)
        t5 = time.perf_counter()

        decoded_by_key[key] = lanes
        row = {
            "model": model_info["name"],
            "family": model_info.get("family", "baseline"),
            "frame_idx": int(idx),
            "key": key,
            "image_rel": rec["image_rel"],
            "lane_count": len(lanes),
            "raw_mode": drive["raw_mode"],
            "effective_mode": drive["effective_mode"],
            "feature_count": drive["feature_count"],
            "candidate_count": drive["candidate_count"],
            "measured_center_x": drive["measured_center_x"],
            "measured_heading": drive["measured_heading"],
            "smoothed_center_x": drive["smoothed_center_x"],
            "smoothed_heading": drive["smoothed_heading"],
            "steer_norm": drive["steer_norm"],
            "turn_bias": drive["turn_bias"],
            "lost_frames": drive["lost_frames"],
            "read_ms": (t1 - t0) * 1000,
            "preprocess_ms": (t2 - t1) * 1000,
            "inference_ms": (t3 - t2) * 1000,
            "decode_ms": (t4 - t3) * 1000,
            "steering_ms": (t5 - t4) * 1000,
            "pipeline_ms": (t5 - t1) * 1000,
            "offline_total_ms": (t5 - t0) * 1000,
        }
        rows.append(row)
    return pd.DataFrame(rows), decoded_by_key

all_model_infos = [BASELINE] + CANDIDATES
sequence_tables = {}
decoded_tables = {}

for item in all_model_infos:
    print("running:", item["name"])
    df, dec = run_model_sequence(item, seq_records)
    sequence_tables[item["name"]] = df
    decoded_tables[item["name"]] = dec
    print(item["name"], "pipeline mean ms:", df["pipeline_ms"].mean(), "fps:", 1000.0 / df["pipeline_ms"].mean())

seq_all_df = pd.concat(sequence_tables.values(), ignore_index=True)
seq_all_df.to_csv(TABLE_DIR / "sequence_outputs_all_models.csv", index=False, encoding="utf-8-sig")
display(seq_all_df.groupby("model")[["pipeline_ms", "preprocess_ms", "inference_ms", "decode_ms", "steering_ms"]].mean())


running: original_fp32
original_fp32 pipeline mean ms: 71.08559749996554 fps: 14.067547227136759
running: ptq11_static_qdq_u8s8
ptq11_static_qdq_u8s8 pipeline mean ms: 36.087974583263836 fps: 27.710061635428
running: ptq11_static_qoperator_u8s8
ptq11_static_qoperator_u8s8 pipeline mean ms: 40.727075416710555 fps: 24.553690383319157
running: ptq11b_backbone_layer4_qdq_u8s8
ptq11b_backbone_layer4_qdq_u8s8 pipeline mean ms: 55.08668416658414 fps: 18.153207351815976
running: ptq11b_backbone_all_qdq_u8s8
ptq11b_backbone_all_qdq_u8s8 pipeline mean ms: 35.82205124988226 fps: 27.915765990739764
running: ptq11b_backbone_neck_qdq_u8s8
ptq11b_backbone_neck_qdq_u8s8 pipeline mean ms: 36.742217500038045 fps: 27.216647988079774
running: ptq11b_backbone_all_qoperator_u8s8
ptq11b_backbone_all_qoperator_u8s8 pipeline mean ms: 43.642437500087304 fps: 22.91347727766121
running: qat15_layer4_only_static_qdq_u8s8
qat15_layer4_only_static_qdq_u8s8 pipeline mean ms: 36.80789666655073 fps: 27.168083225705
run

,pipeline_ms,preprocess_ms,inference_ms,decode_ms,steering_ms
model,,,,,
original_fp32,71.085597,2.917913,67.326594,0.631662,0.209428
ptq11_static_qdq_u8s8,36.087975,2.960152,32.331728,0.644620,0.151475
ptq11_static_qoperator_u8s8,40.727075,2.899147,37.052683,0.615334,0.159911
ptq11b_backbone_all_qdq_u8s8,35.822051,2.872604,32.238955,0.563396,0.147096
ptq11b_backbone_all_qoperator_u8s8,43.642438,3.072695,39.861457,0.567352,0.140934
ptq11b_backbone_layer4_qdq_u8s8,55.086684,2.787038,51.511904,0.587843,0.199900
ptq11b_backbone_neck_qdq_u8s8,36.742218,2.984222,32.935938,0.607085,0.214973
qat15_full_model_static_qdq_u8s8,38.589219,3.041441,34.835070,0.562001,0.150706
qat15_layer4_only_static_qdq_u8s8,36.807897,2.909015,33.129076,0.614615,0.155190


In [8]:
def latency_summary(df):
    rows = []
    for model, g in df.groupby("model"):
        rows.append({
            "model": model,
            "frames": len(g),
            "pipeline_mean_ms": float(g["pipeline_ms"].mean()),
            "pipeline_p50_ms": float(g["pipeline_ms"].quantile(0.50)),
            "pipeline_p95_ms": float(g["pipeline_ms"].quantile(0.95)),
            "preprocess_mean_ms": float(g["preprocess_ms"].mean()),
            "inference_mean_ms": float(g["inference_ms"].mean()),
            "decode_mean_ms": float(g["decode_ms"].mean()),
            "steering_mean_ms": float(g["steering_ms"].mean()),
            "fps_from_pipeline_mean": float(1000.0 / g["pipeline_ms"].mean()),
        })
    out = pd.DataFrame(rows).sort_values("pipeline_mean_ms")
    base_ms = float(out[out["model"] == "original_fp32"]["pipeline_mean_ms"].iloc[0])
    out["speedup_vs_original_fp32"] = base_ms / out["pipeline_mean_ms"]
    return out

lat_df = latency_summary(seq_all_df)
lat_df.to_csv(TABLE_DIR / "latency_summary.csv", index=False, encoding="utf-8-sig")
display(lat_df)


,model,frames,pipeline_mean_ms,pipeline_p50_ms,pipeline_p95_ms,preprocess_mean_ms,inference_mean_ms,decode_mean_ms,steering_mean_ms,fps_from_pipeline_mean,speedup_vs_original_fp32
3,ptq11b_backbone_all_qdq_u8s8,240,35.822051,32.15845,63.526555,2.872604,32.238955,0.563396,0.147096,27.915766,1.984409
1,ptq11_static_qdq_u8s8,240,36.087975,32.52150,61.239080,2.960152,32.331728,0.644620,0.151475,27.710062,1.969786
6,ptq11b_backbone_neck_qdq_u8s8,240,36.742218,33.01770,61.380410,2.984222,32.935938,0.607085,0.214973,27.216648,1.934712
8,qat15_layer4_only_static_qdq_u8s8,240,36.807897,32.40510,65.320510,2.909015,33.129076,0.614615,0.155190,27.168083,1.931259
7,qat15_full_model_static_qdq_u8s8,240,38.589219,34.04170,70.958535,3.041441,34.835070,0.562001,0.150706,25.913974,1.842110
2,ptq11_static_qoperator_u8s8,240,40.727075,36.84095,63.420120,2.899147,37.052683,0.615334,0.159911,24.553690,1.745414
4,ptq11b_backbone_all_qoperator_u8s8,240,43.642438,39.39385,70.482090,3.072695,39.861457,0.567352,0.140934,22.913477,1.628818
5,ptq11b_backbone_layer4_qdq_u8s8,240,55.086684,48.04560,97.470680,2.787038,51.511904,0.587843,0.199900,18.153207,1.290432
0,original_fp32,240,71.085597,62.35800,130.903295,2.917913,67.326594,0.631662,0.209428,14.067547,1.000000


## Driving behavior 비교

여기서는 원본 FP32와 후보의 최종 driving output을 비교한다.

작은 값 근처의 부호 변화는 무시한다.

- `strong_steer_sign_flip`: 원본이 확실히 좌/우로 조향하는데 후보가 반대로 조향
- `strong_heading_sign_flip`: 원본 tangent heading이 확실한데 후보가 반대 heading
- `mode_mismatch`: `both_stable`, `single_left/right`, `lost/recovery` 판단 차이
- `candidate_extra_large_jump`: 후보만 갑자기 큰 조향 점프를 만든 경우


In [9]:
def sign_with_deadband(x, th):
    if pd.isna(x):
        return 0
    if x > th:
        return 1
    if x < -th:
        return -1
    return 0

def coarse_mode(mode):
    if not isinstance(mode, str):
        return "unknown"
    if "single_left" in mode:
        return "single_left"
    if "single_right" in mode:
        return "single_right"
    if "both" in mode:
        return "both"
    if "lost" in mode:
        return "lost"
    if "unstable" in mode:
        return "unstable"
    return mode

def compare_to_baseline(base_df, cand_df, cand_name):
    cols = [
        "frame_idx", "key", "image_rel", "lane_count", "raw_mode", "effective_mode",
        "measured_center_x", "measured_heading", "smoothed_center_x", "smoothed_heading", "steer_norm",
    ]
    b = base_df[cols].copy().add_prefix("base_")
    c = cand_df[cols].copy().add_prefix("cand_")
    merged = pd.concat([b, c], axis=1)
    merged["candidate"] = cand_name
    merged["frame_idx"] = merged["base_frame_idx"]
    merged["key"] = merged["base_key"]
    merged["image_rel"] = merged["base_image_rel"]
    merged["steer_abs_diff"] = (merged["cand_steer_norm"] - merged["base_steer_norm"]).abs()
    merged["heading_abs_diff"] = (merged["cand_smoothed_heading"] - merged["base_smoothed_heading"]).abs()
    merged["center_abs_diff_px"] = (merged["cand_smoothed_center_x"] - merged["base_smoothed_center_x"]).abs()
    merged["lane_count_mismatch"] = merged["cand_lane_count"] != merged["base_lane_count"]
    merged["mode_mismatch"] = merged["cand_effective_mode"] != merged["base_effective_mode"]
    merged["base_coarse_mode"] = merged["base_effective_mode"].map(coarse_mode)
    merged["cand_coarse_mode"] = merged["cand_effective_mode"].map(coarse_mode)
    merged["coarse_mode_mismatch"] = merged["base_coarse_mode"] != merged["cand_coarse_mode"]
    merged["single_side_flip"] = (
        ((merged["base_coarse_mode"] == "single_left") & (merged["cand_coarse_mode"] == "single_right"))
        | ((merged["base_coarse_mode"] == "single_right") & (merged["cand_coarse_mode"] == "single_left"))
    )
    merged["lost_disagreement"] = (merged["base_coarse_mode"] == "lost") != (merged["cand_coarse_mode"] == "lost")

    merged["base_steer_sign"] = merged["base_steer_norm"].apply(lambda v: sign_with_deadband(v, STRONG_STEER_TH))
    merged["cand_steer_sign"] = merged["cand_steer_norm"].apply(lambda v: sign_with_deadband(v, STRONG_STEER_TH))
    merged["strong_steer_sign_flip"] = (merged["base_steer_sign"] != 0) & (merged["cand_steer_sign"] != 0) & (merged["base_steer_sign"] != merged["cand_steer_sign"])

    merged["base_heading_sign"] = merged["base_smoothed_heading"].apply(lambda v: sign_with_deadband(v, STRONG_HEADING_TH))
    merged["cand_heading_sign"] = merged["cand_smoothed_heading"].apply(lambda v: sign_with_deadband(v, STRONG_HEADING_TH))
    merged["strong_heading_sign_flip"] = (merged["base_heading_sign"] != 0) & (merged["cand_heading_sign"] != 0) & (merged["base_heading_sign"] != merged["cand_heading_sign"])

    merged["base_steer_jump"] = merged["base_steer_norm"].diff().abs().fillna(0.0)
    merged["cand_steer_jump"] = merged["cand_steer_norm"].diff().abs().fillna(0.0)
    merged["candidate_extra_large_jump"] = (merged["cand_steer_jump"] > LARGE_STEER_JUMP_TH) & (merged["base_steer_jump"] <= LARGE_STEER_JUMP_TH)

    merged["risk_score"] = (
        merged["strong_steer_sign_flip"].astype(int) * 100
        + merged["single_side_flip"].astype(int) * 90
        + merged["strong_heading_sign_flip"].astype(int) * 70
        + merged["candidate_extra_large_jump"].astype(int) * 50
        + merged["lost_disagreement"].astype(int) * 20
        + merged["coarse_mode_mismatch"].astype(int) * 12
        + merged["mode_mismatch"].astype(int) * 4
        + merged["lane_count_mismatch"].astype(int) * 5
        + merged["steer_abs_diff"] * 30
        + merged["heading_abs_diff"] * 20
    )
    return merged

base_df = sequence_tables["original_fp32"].reset_index(drop=True)
diff_tables = []
for item in CANDIDATES:
    cand_df = sequence_tables[item["name"]].reset_index(drop=True)
    diff_tables.append(compare_to_baseline(base_df, cand_df, item["name"]))

diff_df = pd.concat(diff_tables, ignore_index=True)
diff_df.to_csv(TABLE_DIR / "frame_behavior_diffs.csv", index=False, encoding="utf-8-sig")
display(diff_df.head())


,base_frame_idx,base_key,base_image_rel,base_lane_count,base_raw_mode,base_effective_mode,base_measured_center_x,base_measured_heading,base_smoothed_center_x,base_smoothed_heading,...,base_steer_sign,cand_steer_sign,strong_steer_sign_flip,base_heading_sign,cand_heading_sign,strong_heading_sign_flip,base_steer_jump,cand_steer_jump,candidate_extra_large_jump,risk_score
0,0,f3s_0000_c0bbf9d393,i/f3s_0000_c0bbf9d393.jpg,2,both_stable,both_stable,550.553733,0.035062,599.276866,0.017531,...,0,0,False,0,0,False,0.000000,0.000000,False,0.115476
1,1,f3s_0001_ad507632de,i/f3s_0001_ad507632de.jpg,2,both_stable,both_stable,549.661966,-0.026979,574.469416,-0.004724,...,0,0,False,0,0,False,0.026937,0.027655,False,0.021622
2,2,f3s_0002_5001e69f43,i/f3s_0002_5001e69f43.jpg,2,single_right,single_right,672.597860,-0.495526,623.533638,-0.250125,...,0,0,False,-1,-1,False,0.044833,0.046535,False,0.163841
3,3,f3s_0003_ab11851eee,i/f3s_0003_ab11851eee.jpg,2,both_stable,both_stable,669.404299,0.222616,646.468968,-0.013755,...,0,0,False,0,0,False,0.041916,0.042125,False,0.099784
4,4,f3s_0004_0befa0c953,i/f3s_0004_0befa0c953.jpg,2,both_stable,both_stable,574.948422,0.028385,610.708695,0.007315,...,0,0,False,0,0,False,0.008315,0.011355,False,0.221840


In [10]:
def behavior_summary(diff_df, latency_df):
    rows = []
    for cand, g in diff_df.groupby("candidate"):
        lat = latency_df[latency_df["model"] == cand].iloc[0].to_dict()
        frames = len(g)
        row = {
            "candidate": cand,
            "frames": frames,
            "pipeline_mean_ms": lat["pipeline_mean_ms"],
            "fps_from_pipeline_mean": lat["fps_from_pipeline_mean"],
            "speedup_vs_original_fp32": lat["speedup_vs_original_fp32"],
            "strong_steer_sign_flip_count": int(g["strong_steer_sign_flip"].sum()),
            "strong_heading_sign_flip_count": int(g["strong_heading_sign_flip"].sum()),
            "single_side_flip_count": int(g["single_side_flip"].sum()),
            "lost_disagreement_count": int(g["lost_disagreement"].sum()),
            "coarse_mode_mismatch_count": int(g["coarse_mode_mismatch"].sum()),
            "mode_mismatch_count": int(g["mode_mismatch"].sum()),
            "mode_mismatch_ratio": float(g["mode_mismatch"].mean()),
            "lane_count_mismatch_count": int(g["lane_count_mismatch"].sum()),
            "candidate_extra_large_jump_count": int(g["candidate_extra_large_jump"].sum()),
            "steer_abs_diff_mean": float(g["steer_abs_diff"].mean()),
            "steer_abs_diff_p95": float(g["steer_abs_diff"].quantile(0.95)),
            "steer_abs_diff_max": float(g["steer_abs_diff"].max()),
            "heading_abs_diff_mean": float(g["heading_abs_diff"].mean()),
            "heading_abs_diff_p95": float(g["heading_abs_diff"].quantile(0.95)),
            "heading_abs_diff_max": float(g["heading_abs_diff"].max()),
            "center_abs_diff_p95_px": float(g["center_abs_diff_px"].quantile(0.95)),
            "risk_score_sum": float(g["risk_score"].sum()),
            "risk_score_max": float(g["risk_score"].max()),
        }
        row["driving_behavior_pass"] = (
            row["strong_steer_sign_flip_count"] == 0
            and row["single_side_flip_count"] == 0
            and row["strong_heading_sign_flip_count"] <= 1
            and row["steer_abs_diff_p95"] <= STEER_ABS_DIFF_P95_LIMIT
            and row["heading_abs_diff_p95"] <= HEADING_ABS_DIFF_P95_LIMIT
            and row["mode_mismatch_ratio"] <= MODE_MISMATCH_RATIO_LIMIT
            and row["candidate_extra_large_jump_count"] <= EXTRA_LARGE_JUMP_LIMIT
        )
        row["speed_priority_pass"] = row["pipeline_mean_ms"] <= 50.0 or row["speedup_vs_original_fp32"] >= 1.25
        row["selected_for_pi_probe"] = bool(row["driving_behavior_pass"] and row["speed_priority_pass"])
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["selected_for_pi_probe", "driving_behavior_pass", "pipeline_mean_ms"], ascending=[False, False, True])

summary_df = behavior_summary(diff_df, lat_df)
summary_df.to_csv(TABLE_DIR / "behavior_gate_summary.csv", index=False, encoding="utf-8-sig")
display(summary_df)


,candidate,frames,pipeline_mean_ms,fps_from_pipeline_mean,speedup_vs_original_fp32,strong_steer_sign_flip_count,strong_heading_sign_flip_count,single_side_flip_count,lost_disagreement_count,coarse_mode_mismatch_count,...,steer_abs_diff_max,heading_abs_diff_mean,heading_abs_diff_p95,heading_abs_diff_max,center_abs_diff_p95_px,risk_score_sum,risk_score_max,driving_behavior_pass,speed_priority_pass,selected_for_pi_probe
2,ptq11b_backbone_all_qdq_u8s8,240,35.822051,27.915766,1.984409,0,0,0,1,3,...,0.124618,0.015681,0.074737,0.391183,10.808664,207.402938,52.505083,True,True,True
5,ptq11b_backbone_neck_qdq_u8s8,240,36.742218,27.216648,1.934712,0,0,0,1,3,...,0.124643,0.015780,0.075062,0.391586,10.674262,208.135301,52.521971,True,True,True
3,ptq11b_backbone_all_qoperator_u8s8,240,43.642438,22.913477,1.628818,0,0,0,1,4,...,0.121574,0.016800,0.113141,0.372831,11.721073,240.688906,51.879826,True,True,True
4,ptq11b_backbone_layer4_qdq_u8s8,240,55.086684,18.153207,1.290432,0,0,0,1,2,...,0.050746,0.003733,0.008853,0.277627,0.304601,89.485232,42.014387,True,True,True
0,ptq11_static_qdq_u8s8,240,36.087975,27.710062,1.969786,0,1,1,8,11,...,0.167132,0.030917,0.171912,0.417620,78.585874,860.730363,110.266032,False,True,False
7,qat15_layer4_only_static_qdq_u8s8,240,36.807897,27.168083,1.931259,0,0,0,5,11,...,0.188560,0.039064,0.222837,0.486931,51.068683,681.251398,49.041198,False,True,False
6,qat15_full_model_static_qdq_u8s8,240,38.589219,25.913974,1.842110,0,1,0,6,11,...,0.163478,0.034806,0.202439,0.430717,57.094122,720.723322,95.550380,False,True,False
1,ptq11_static_qoperator_u8s8,240,40.727075,24.553690,1.745414,0,0,1,6,16,...,0.167616,0.037957,0.198390,0.437993,106.102776,854.703612,110.536670,False,True,False


## 위험 프레임 확인

평균값만 보면 위험 프레임 몇 개가 숨어버릴 수 있다.  
아래 셀은 각 후보에서 `risk_score`가 큰 프레임을 뽑아 overlay sheet로 저장한다.

색상:

- 초록: 원본 FP32 decoded lane
- 분홍: 후보 INT8 decoded lane

이 이미지는 “평균적으로 괜찮다”가 아니라, **가장 위험해 보이는 순간에도 주행 방향성이 버틸 수 있는지**를 보기 위한 것이다.


In [11]:
def draw_lanes_on_image(bgr, lanes, color, thickness=2):
    out = bgr.copy()
    for lane in lanes:
        pts = np.array(lane["points"], dtype=np.float32)
        valid = pts[:, 0] >= 0
        pts = pts[valid]
        if len(pts) < 2:
            continue
        ipts = np.round(pts).astype(np.int32)
        for a, b in zip(ipts[:-1], ipts[1:]):
            cv2.line(out, tuple(a), tuple(b), color, thickness, cv2.LINE_AA)
    return out

def make_top_risk_sheet(candidate, top_k=12):
    g = diff_df[diff_df["candidate"] == candidate].sort_values("risk_score", ascending=False).head(top_k).copy()
    tiles = []
    for _, row in g.iterrows():
        bgr = imread_bgr(PACKAGE_ROOT / row["image_rel"])
        h, w = bgr.shape[:2]
        base_lanes = decoded_tables["original_fp32"][row["key"]]
        cand_lanes = decoded_tables[candidate][row["key"]]
        vis = draw_lanes_on_image(bgr, base_lanes, (0, 220, 0), 3)
        vis = draw_lanes_on_image(vis, cand_lanes, (255, 0, 255), 2)
        text1 = f"{candidate}  f={int(row['frame_idx'])} risk={row['risk_score']:.1f}"
        text2 = f"steer {row['base_steer_norm']:+.3f}->{row['cand_steer_norm']:+.3f} diff={row['steer_abs_diff']:.3f}"
        text3 = f"mode {row['base_effective_mode']}->{row['cand_effective_mode']} heading {row['base_smoothed_heading']:+.3f}->{row['cand_smoothed_heading']:+.3f}"
        cv2.rectangle(vis, (0, 0), (w, 82), (0, 0, 0), -1)
        cv2.putText(vis, text1, (12, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,255,255), 2, cv2.LINE_AA)
        cv2.putText(vis, text2, (12, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (255,255,255), 1, cv2.LINE_AA)
        cv2.putText(vis, text3, (12, 74), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (255,255,255), 1, cv2.LINE_AA)
        tiles.append(vis)
    if not tiles:
        return None
    # 3 columns sheet
    cols = 3
    rows = int(math.ceil(len(tiles) / cols))
    tile_h, tile_w = tiles[0].shape[:2]
    sheet = np.zeros((rows * tile_h, cols * tile_w, 3), dtype=np.uint8)
    for i, tile in enumerate(tiles):
        r, c = divmod(i, cols)
        sheet[r*tile_h:(r+1)*tile_h, c*tile_w:(c+1)*tile_w] = tile
    out_path = VIS_DIR / f"{candidate}_top_risk_overlay.jpg"
    imwrite_bgr(out_path, sheet)
    return out_path

top_rows = []
for cand in [c["name"] for c in CANDIDATES]:
    out = make_top_risk_sheet(cand, top_k=12)
    print(cand, out)
    top_rows.append(diff_df[diff_df["candidate"] == cand].sort_values("risk_score", ascending=False).head(12))

top_risk_df = pd.concat(top_rows, ignore_index=True)
top_risk_df.to_csv(TABLE_DIR / "top_risk_frames.csv", index=False, encoding="utf-8-sig")
display(top_risk_df[["candidate", "frame_idx", "risk_score", "base_steer_norm", "cand_steer_norm", "steer_abs_diff", "base_effective_mode", "cand_effective_mode", "strong_steer_sign_flip", "strong_heading_sign_flip"]])


ptq11_static_qdq_u8s8 ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\16_quantized_driving_behavior_gate\review_outputs\00_speed_first_driving_behavior_gate_v1\visuals\ptq11_static_qdq_u8s8_top_risk_overlay.jpg
ptq11_static_qoperator_u8s8 ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\16_quantized_driving_behavior_gate\review_outputs\00_speed_first_driving_behavior_gate_v1\visuals\ptq11_static_qoperator_u8s8_top_risk_overlay.jpg
ptq11b_backbone_layer4_qdq_u8s8 ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\16_quantized_driving_behavior_gate\review_outputs\00_speed_first_driving_behavior_gate_v1\visuals\ptq11b_backbone_layer4_qdq_u8s8_top_risk_overlay.jpg
ptq11b_backbone_all_qdq_u8s8 ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\16_quantized_driving_behavior_gate\review_outputs\00_speed_first_driving_behavior_gate_v1\visuals\ptq11b_backbone_all_qdq_

,candidate,frame_idx,risk_score,base_steer_norm,cand_steer_norm,steer_abs_diff,base_effective_mode,cand_effective_mode,strong_steer_sign_flip,strong_heading_sign_flip
0,ptq11_static_qdq_u8s8,102,110.266032,-0.005430,-0.144947,0.139517,single_left,single_right,False,False
1,ptq11_static_qdq_u8s8,220,96.636784,0.209586,0.115852,0.093734,unstable_blend,single_left,False,True
2,ptq11_static_qdq_u8s8,224,54.366374,0.169877,0.002745,0.167132,lost_short_recovery,single_left,False,False
3,ptq11_static_qdq_u8s8,152,51.652832,0.109064,0.219121,0.110057,lost_short_recovery,unstable_blend,False,False
4,ptq11_static_qdq_u8s8,12,46.397846,-0.120635,-0.055040,0.065595,unstable_blend,lost_short_recovery,False,False
...,...,...,...,...,...,...,...,...,...,...
91,qat15_full_model_static_qdq_u8s8,33,25.294461,-0.131556,-0.295034,0.163478,unstable_blend,single_right,False,False
92,qat15_full_model_static_qdq_u8s8,20,24.423354,-0.076849,-0.173469,0.096620,unstable_blend,single_right,False,False
93,qat15_full_model_static_qdq_u8s8,32,21.167662,-0.090924,-0.128819,0.037895,unstable_blend,single_right,False,False
94,qat15_full_model_static_qdq_u8s8,179,20.252935,-0.068655,-0.083004,0.014349,both_stable,unstable_blend,False,False


In [12]:
report = {
    "notebook": "00_speed_first_driving_behavior_gate_v1",
    "purpose": "Re-evaluate fast quantized candidates with driving behavior gates rather than strict raw/lane parity.",
    "baseline": str(BASELINE["path"]),
    "candidate_scope": CANDIDATE_SCOPE,
    "candidates": [{"name": c["name"], "family": c["family"], "path": str(c["path"])} for c in CANDIDATES],
    "contract_sources": {
        "package_root": str(PACKAGE_ROOT),
        "decode_contract": str(PACKAGE_ROOT / "c" / "decode_contract_v1.json"),
        "driving_contract": str(PACKAGE_ROOT / "c" / "driving_contract_v1.json"),
        "records_manifest": str(PACKAGE_ROOT / "t" / "records_manifest.csv"),
    },
    "gate_thresholds": {
        "strong_steer_threshold": STRONG_STEER_TH,
        "strong_heading_threshold": STRONG_HEADING_TH,
        "steer_abs_diff_p95_limit": STEER_ABS_DIFF_P95_LIMIT,
        "heading_abs_diff_p95_limit": HEADING_ABS_DIFF_P95_LIMIT,
        "mode_mismatch_ratio_limit": MODE_MISMATCH_RATIO_LIMIT,
        "extra_large_jump_limit": EXTRA_LARGE_JUMP_LIMIT,
        "large_steer_jump_threshold": LARGE_STEER_JUMP_TH,
        "single_side_flip_required": 0,
        "strong_steer_sign_flip_required": 0,
    },
    "selected_for_pi_probe": summary_df[summary_df["selected_for_pi_probe"]]["candidate"].tolist(),
    "manual_review_required": True,
    "outputs": {
        "tables": str(TABLE_DIR),
        "visuals": str(VIS_DIR),
    },
}
write_json(OUT_ROOT / "driving_behavior_gate_report.json", report)
print(json.dumps(report, indent=2, ensure_ascii=False))


{
  "notebook": "00_speed_first_driving_behavior_gate_v1",
  "purpose": "Re-evaluate fast quantized candidates with driving behavior gates rather than strict raw/lane parity.",
  "baseline": "~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\10_experiments\\12_clrkdnet_supervised_rebuild\\review_outputs\\09_onnx_export_parity_v1\\models\\MapLane_LocalFit_Field12_v1_best_opset17_static_b1.onnx",
  "candidate_scope": "speed_shortlist_v2",
  "candidates": [
    {
      "name": "ptq11_static_qdq_u8s8",
      "family": "11_full_ptq",
      "path": "~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\10_experiments\\12_clrkdnet_supervised_rebuild\\review_outputs\\11_onnx_cpu_quantization_candidates_v1\\models\\static_qdq_u8s8.onnx"
    },
    {
      "name": "ptq11_static_qoperator_u8s8",
      "family": "11_full_ptq",
      "path": "~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\10_experiments\\12_clrkdnet_supervised_rebuild\\r

## 해석 방법

1. `latency_summary.csv`에서 실제 속도 이득을 먼저 본다.
2. `behavior_gate_summary.csv`에서 `selected_for_pi_probe=True`인 후보가 있는지 본다.
3. 후보가 없더라도 `strong_steer_sign_flip_count=0`이고 `steer_abs_diff_p95`가 낮은 후보는 실험 후보로 남길 수 있다.
4. 반드시 `visuals/*_top_risk_overlay.jpg`를 확인한다. 평균은 괜찮아도 top-risk frame에서 반대 조향이 나오면 위험하다.

이 노트북의 결과는 Pi 배포 확정이 아니다.  
여기서 고른 후보만 16 또는 다음 폴더의 Pi probe로 보내서 `threads=2/3`, voltage, temperature, 실제 주행을 확인한다.
